# 설비 고장 예측 — EDA와 모델 학습 (AI4I 2020)

- 데이터: 밀링 머신 운전 기록 (10,000행 · 14열)
- 목표: 고장 여부(Machine failure) 예측 — 이진분류
- 흐름: 불러오기 → 학습 전 확인 → EDA → 학습 → 해석
- 참고: 데이터 소개 data_AI4I2020.txt

- 이 데이터의 핵심: **고장 유형 5종이 누수** · 불균형(3.4%)

## 1. 불러오기

- 일반 CSV · 결측 없음
- 컬럼명에 단위 포함(Air temperature [K] 등) → 정리 권장

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("AI4I2020.csv")
print(df.shape)          # (10000, 14)
df.head()

In [ ]:
# 컬럼명 정리 (단위·공백 제거)
df.columns = ["UDI","product_id","type","air_temp","process_temp",
              "rot_speed","torque","tool_wear","machine_failure",
              "TWF","HDF","PWF","OSF","RNF"]
print(df.columns.tolist())

## 2. 학습 전 확인

- 이 데이터의 핵심 판단은 **고장 유형 5종**

### 2-1. 고장 유형 5종이 누수 (가장 중요)

- TWF/HDF/PWF/OSF/RNF = 구체적 고장 원인 표시
- 이 중 하나라도 1이면 대체로 machine_failure=1
- 즉 '고장 원인'으로 '고장 여부'를 맞히는 것 = 누수
- 실전에선 고장 나기 전에 예측해야 하므로 원인을 미리 알 수 없음

In [ ]:
# 누수 확인: 유형 중 하나라도 1이면 고장인가
anyfail = df[["TWF","HDF","PWF","OSF","RNF"]].sum(axis=1) > 0
print("유형 하나라도 1인 행:", anyfail.sum())
print("그 중 고장:", (df.loc[anyfail,"machine_failure"]==1).sum())
# 거의 일치 → 누수

# 제거: UDI(식별자), product_id, 고장유형 5종
df = df.drop(columns=["UDI","product_id",
                      "TWF","HDF","PWF","OSF","RNF"])
print("제거 후:", df.shape)

### 2-2. product_id 앞글자 = type (중복 정보)

- product_id는 "M14860"처럼 앞글자가 type(L/M/H)과 같음
- 위에서 product_id는 이미 제거함 (type만 유지)

### 2-3. 불균형 (고장 3.4%)

- 고장이 3.4%뿐 → "전부 정상"만 해도 정확도 96.6%
- 정확도 대신 roc_auc로 평가해야 의미 있음

In [ ]:
print("고장률:", (df["machine_failure"]==1).mean().round(3))
# → eval_metric="roc_auc" 사용 (아래 4절)

## 3. EDA

In [ ]:
from data_profiling import ProfileReport

profile = ProfileReport(df, progress_bar=False)
profile.to_file("ai4i_eda.html")   # 브라우저에서 열기

### 3-1. 운전 조건과 고장

- 토크·회전속도·공구마모가 고장과 관련이 있는가

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(13,3))
for i, c in enumerate(["torque","rot_speed","tool_wear"]):
    df.boxplot(column=c, by="machine_failure", ax=ax[i])
plt.suptitle("")
plt.show()

## 4. 물리 파생변수 (선택 심화)

- 물리적으로 의미 있는 변수를 만들 수 있음
  - power = 토크 × 회전속도 (기계적 출력)
  - temp_diff = 공정온도 − 공기온도 (발열)
- 도메인 지식으로 만드는 변수 → 성능에 도움될 수 있음

In [ ]:
# 회전속도(rpm) → 각속도(rad/s) 변환 후 출력 계산
df["power"] = df["torque"] * df["rot_speed"] * 2*np.pi/60
df["temp_diff"] = df["process_temp"] - df["air_temp"]
print(df[["power","temp_diff"]].describe().round(2))

## 5. 학습

- 불균형 → roc_auc
- 결측·인코딩(type L/M/H)은 내부 자동

In [ ]:
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    df, test_size=0.2, random_state=42,
    stratify=df["machine_failure"])

predictor = TabularPredictor(
    label="machine_failure",
    eval_metric="roc_auc",
).fit(train_data, presets="medium_quality", time_limit=300)

In [ ]:
predictor.leaderboard(test_data)

## 6. 해석

In [ ]:
print(predictor.evaluate(test_data))

### 6-1. 변수 중요도

- 무엇이 고장을 예측하는가
- 파생변수(power, temp_diff)가 원본보다 유용한가

In [ ]:
predictor.feature_importance(test_data)

## 정리

- 고장 유형 5종 = 누수 → 제거 (실전에선 원인을 미리 모름)
- 식별자(UDI·product_id) 제거
- 불균형 3.4% → roc_auc (정확도는 무의미)
- 물리 파생변수(power·temp_diff) → 도메인 지식
- 학습·앙상블은 TabularPredictor가 자동

- Heart Failure와 같은 교훈: 누수 판단은 사람 몫
  → 성능이 100%에 가까우면 오히려 누수를 의심할 것